# Customer Churn Prediction: Model Development

This notebook focuses on developing and evaluating machine learning models for predicting customer churn.

## 1. Setup and Data Loading

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve, precision_recall_curve, average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
import xgboost as xgb
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
import sys
import os
from pathlib import Path
import joblib
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')

# Configure plot size and resolution
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100

# Display all columns
pd.set_option('display.max_columns', None)

In [ ]:
# Add project root to path
project_root = Path().resolve().parent
sys.path.append(str(project_root))

# Import project modules
from src.data.data_loader import load_raw_data, preprocess_data, split_data
from src.features.feature_engineering import FeatureEngineer
from src.config import MODELS_DIR, RANDOM_STATE

In [ ]:
# Load and preprocess data
raw_data = load_raw_data()
processed_data = preprocess_data(raw_data)

# Split data
X_train, X_val, X_test, y_train, y_val, y_test = split_data(processed_data)

print(f&quot;Training set: {X_train.shape[0]} samples&quot;)
print(f&quot;Validation set: {X_val.shape[0]} samples&quot;)
print(f&quot;Test set: {X_test.shape[0]} samples&quot;)

## 2. Feature Engineering

In [ ]:
# Initialize feature engineer
feature_engineer = FeatureEngineer()

# Create additional features
X_train = feature_engineer.create_interaction_features(X_train)
X_train = feature_engineer.create_service_count_feature(X_train)

X_val = feature_engineer.create_interaction_features(X_val)
X_val = feature_engineer.create_service_count_feature(X_val)

X_test = feature_engineer.create_interaction_features(X_test)
X_test = feature_engineer.create_service_count_feature(X_test)

# Display new features
new_features = [col for col in X_train.columns if col not in processed_data.columns]
print(f&quot;New features created: {new_features}&quot;)

# Transform features
X_train_transformed = feature_engineer.fit_transform(X_train)
X_val_transformed = feature_engineer.transform(X_val)
X_test_transformed = feature_engineer.transform(X_test)

print(f&quot;Transformed training data shape: {X_train_transformed.shape}&quot;)
print(f&quot;Transformed validation data shape: {X_val_transformed.shape}&quot;)
print(f&quot;Transformed test data shape: {X_test_transformed.shape}&quot;)

## 3. Baseline Models

In [ ]:
# Define evaluation function
def evaluate_model(model, X_train, X_val, y_train, y_val):
    # Train the model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    
    # Get probability predictions
    y_train_prob = model.predict_proba(X_train)[:, 1]
    y_val_prob = model.predict_proba(X_val)[:, 1]
    
    # Calculate metrics
    train_metrics = {
        'accuracy': accuracy_score(y_train, y_train_pred),
        'precision': precision_score(y_train, y_train_pred),
        'recall': recall_score(y_train, y_train_pred),
        'f1': f1_score(y_train, y_train_pred),
        'roc_auc': roc_auc_score(y_train, y_train_prob),
        'avg_precision': average_precision_score(y_train, y_train_prob)
    }
    
    val_metrics = {
        'accuracy': accuracy_score(y_val, y_val_pred),
        'precision': precision_score(y_val, y_val_pred),
        'recall': recall_score(y_val, y_val_pred),
        'f1': f1_score(y_val, y_val_pred),
        'roc_auc': roc_auc_score(y_val, y_val_prob),
        'avg_precision': average_precision_score(y_val, y_val_prob)
    }
    
    return train_metrics, val_metrics

In [ ]:
# Define baseline models
models = {
    'Logistic Regression': LogisticRegression(random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE),
    'XGBoost': xgb.XGBClassifier(random_state=RANDOM_STATE),
    'LightGBM': LGBMClassifier(random_state=RANDOM_STATE),
    'CatBoost': CatBoostClassifier(random_state=RANDOM_STATE, verbose=0)
}

# Train and evaluate baseline models
results = {}

for name, model in models.items():
    print(f&quot;Training {name}...&quot;)
    train_metrics, val_metrics = evaluate_model(model, X_train_transformed, X_val_transformed, y_train, y_val)
    results[name] = {'train': train_metrics, 'val': val_metrics}
    
    print(f&quot;  Training metrics: Accuracy={train_metrics['accuracy']:.4f}, ROC AUC={train_metrics['roc_auc']:.4f}, F1={train_metrics['f1']:.4f}&quot;)
    print(f&quot;  Validation metrics: Accuracy={val_metrics['accuracy']:.4f}, ROC AUC={val_metrics['roc_auc']:.4f}, F1={val_metrics['f1']:.4f}&quot;)
    print()

In [ ]:
# Compare model performance
def plot_model_comparison(results, metric='roc_auc'):
    # Extract metrics
    model_names = list(results.keys())
    train_metrics = [results[name]['train'][metric] for name in model_names]
    val_metrics = [results[name]['val'][metric] for name in model_names]
    
    # Create DataFrame for plotting
    df = pd.DataFrame({
        'Model': model_names,
        'Training': train_metrics,
        'Validation': val_metrics
    })
    
    # Sort by validation metric
    df = df.sort_values('Validation', ascending=False)
    
    # Plot
    plt.figure(figsize=(12, 6))
    x = np.arange(len(df))
    width = 0.35
    
    plt.bar(x - width/2, df['Training'], width, label='Training')
    plt.bar(x + width/2, df['Validation'], width, label='Validation')
    
    plt.xlabel('Model')
    plt.ylabel(f'{metric.upper()}')
    plt.title(f'Model Comparison - {metric.upper()}')
    plt.xticks(x, df['Model'], rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()

# Plot model comparison for different metrics
metrics = ['accuracy', 'roc_auc', 'f1', 'precision', 'recall']
for metric in metrics:
    plot_model_comparison(results, metric)

## 4. Handling Class Imbalance with SMOTE

In [ ]:
# Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train_transformed, y_train)

print(f&quot;Original training set shape: {X_train_transformed.shape}&quot;)
print(f&quot;SMOTE-resampled training set shape: {X_train_smote.shape}&quot;)
print(f&quot;Original class distribution: {pd.Series(y_train).value_counts()}&quot;)
print(f&quot;SMOTE-resampled class distribution: {pd.Series(y_train_smote).value_counts()}&quot;)

In [ ]:
# Train and evaluate models with SMOTE
smote_results = {}

for name, model in models.items():
    print(f&quot;Training {name} with SMOTE...&quot;)
    
    # Train the model on SMOTE-resampled data
    model.fit(X_train_smote, y_train_smote)
    
    # Make predictions
    y_train_pred = model.predict(X_train_transformed)
    y_val_pred = model.predict(X_val_transformed)
    
    # Get probability predictions
    y_train_prob = model.predict_proba(X_train_transformed)[:, 1]
    y_val_prob = model.predict_proba(X_val_transformed)[:, 1]
    
    # Calculate metrics
    train_metrics = {
        'accuracy': accuracy_score(y_train, y_train_pred),
        'precision': precision_score(y_train, y_train_pred),
        'recall': recall_score(y_train, y_train_pred),
        'f1': f1_score(y_train, y_train_pred),
        'roc_auc': roc_auc_score(y_train, y_train_prob),
        'avg_precision': average_precision_score(y_train, y_train_prob)
    }
    
    val_metrics = {
        'accuracy': accuracy_score(y_val, y_val_pred),
        'precision': precision_score(y_val, y_val_pred),
        'recall': recall_score(y_val, y_val_pred),
        'f1': f1_score(y_val, y_val_pred),
        'roc_auc': roc_auc_score(y_val, y_val_prob),
        'avg_precision': average_precision_score(y_val, y_val_prob)
    }
    
    smote_results[name] = {'train': train_metrics, 'val': val_metrics}
    
    print(f&quot;  Training metrics: Accuracy={train_metrics['accuracy']:.4f}, ROC AUC={train_metrics['roc_auc']:.4f}, F1={train_metrics['f1']:.4f}&quot;)
    print(f&quot;  Validation metrics: Accuracy={val_metrics['accuracy']:.4f}, ROC AUC={val_metrics['roc_auc']:.4f}, F1={val_metrics['f1']:.4f}&quot;)
    print()

In [ ]:
# Compare model performance with and without SMOTE
def compare_smote_results(results, smote_results, metric='roc_auc'):
    # Extract metrics
    model_names = list(results.keys())
    val_metrics = [results[name]['val'][metric] for name in model_names]
    smote_val_metrics = [smote_results[name]['val'][metric] for name in model_names]
    
    # Create DataFrame for plotting
    df = pd.DataFrame({
        'Model': model_names,
        'Without SMOTE': val_metrics,
        'With SMOTE': smote_val_metrics
    })
    
    # Sort by SMOTE metric
    df = df.sort_values('With SMOTE', ascending=False)
    
    # Plot
    plt.figure(figsize=(12, 6))
    x = np.arange(len(df))
    width = 0.35
    
    plt.bar(x - width/2, df['Without SMOTE'], width, label='Without SMOTE')
    plt.bar(x + width/2, df['With SMOTE'], width, label='With SMOTE')
    
    plt.xlabel('Model')
    plt.ylabel(f'{metric.upper()}')
    plt.title(f'Model Comparison With and Without SMOTE - {metric.upper()}')
    plt.xticks(x, df['Model'], rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()

# Compare results for different metrics
metrics = ['accuracy', 'roc_auc', 'f1', 'precision', 'recall']
for metric in metrics:
    compare_smote_results(results, smote_results, metric)

## 5. Hyperparameter Tuning

In [ ]:
# Select the best performing model for hyperparameter tuning
# Based on the results, let's tune XGBoost
xgb_model = xgb.XGBClassifier(random_state=RANDOM_STATE)

# Define hyperparameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2]
}

# Use RandomizedSearchCV for efficiency
random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid,
    n_iter=20,
    scoring='roc_auc',
    cv=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

# Fit the random search
random_search.fit(X_train_smote, y_train_smote)

# Print best parameters
print(f&quot;Best parameters: {random_search.best_params_}&quot;)
print(f&quot;Best ROC AUC score: {random_search.best_score_:.4f}&quot;)

In [ ]:
# Train the model with best parameters
best_xgb = xgb.XGBClassifier(
    **random_search.best_params_,
    random_state=RANDOM_STATE
)

# Train on SMOTE-resampled data
best_xgb.fit(X_train_smote, y_train_smote)

# Make predictions
y_train_pred = best_xgb.predict(X_train_transformed)
y_val_pred = best_xgb.predict(X_val_transformed)
y_test_pred = best_xgb.predict(X_test_transformed)

# Get probability predictions
y_train_prob = best_xgb.predict_proba(X_train_transformed)[:, 1]
y_val_prob = best_xgb.predict_proba(X_val_transformed)[:, 1]
y_test_prob = best_xgb.predict_proba(X_test_transformed)[:, 1]

In [ ]:
# Evaluate the best model
print(&quot;Training Set Metrics:&quot;)
print(f&quot;Accuracy: {accuracy_score(y_train, y_train_pred):.4f}&quot;)
print(f&quot;Precision: {precision_score(y_train, y_train_pred):.4f}&quot;)
print(f&quot;Recall: {recall_score(y_train, y_train_pred):.4f}&quot;)
print(f&quot;F1 Score: {f1_score(y_train, y_train_pred):.4f}&quot;)
print(f&quot;ROC AUC: {roc_auc_score(y_train, y_train_prob):.4f}&quot;)
print(f&quot;Average Precision: {average_precision_score(y_train, y_train_prob):.4f}&quot;)
print(&quot;\nClassification Report:&quot;)
print(classification_report(y_train, y_train_pred))

print(&quot;\nValidation Set Metrics:&quot;)
print(f&quot;Accuracy: {accuracy_score(y_val, y_val_pred):.4f}&quot;)
print(f&quot;Precision: {precision_score(y_val, y_val_pred):.4f}&quot;)
print(f&quot;Recall: {recall_score(y_val, y_val_pred):.4f}&quot;)
print(f&quot;F1 Score: {f1_score(y_val, y_val_pred):.4f}&quot;)
print(f&quot;ROC AUC: {roc_auc_score(y_val, y_val_prob):.4f}&quot;)
print(f&quot;Average Precision: {average_precision_score(y_val, y_val_prob):.4f}&quot;)
print(&quot;\nClassification Report:&quot;)
print(classification_report(y_val, y_val_pred))

print(&quot;\nTest Set Metrics:&quot;)
print(f&quot;Accuracy: {accuracy_score(y_test, y_test_pred):.4f}&quot;)
print(f&quot;Precision: {precision_score(y_test, y_test_pred):.4f}&quot;)
print(f&quot;Recall: {recall_score(y_test, y_test_pred):.4f}&quot;)
print(f&quot;F1 Score: {f1_score(y_test, y_test_pred):.4f}&quot;)
print(f&quot;ROC AUC: {roc_auc_score(y_test, y_test_prob):.4f}&quot;)
print(f&quot;Average Precision: {average_precision_score(y_test, y_test_prob):.4f}&quot;)
print(&quot;\nClassification Report:&quot;)
print(classification_report(y_test, y_test_pred))

## 6. Model Evaluation and Visualization

In [ ]:
# Plot confusion matrix
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

# Plot ROC curve
def plot_roc_curve(y_true, y_prob, title):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = roc_auc_score(y_true, y_prob)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.4f}')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title)
    plt.legend(loc='lower right')
    plt.show()

# Plot precision-recall curve
def plot_precision_recall_curve(y_true, y_prob, title):
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    avg_precision = average_precision_score(y_true, y_prob)
    
    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, label=f'AP = {avg_precision:.4f}')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(title)
    plt.legend(loc='upper right')
    plt.show()

In [ ]:
# Plot confusion matrices
plot_confusion_matrix(y_train, y_train_pred, 'Confusion Matrix - Training Set')
plot_confusion_matrix(y_val, y_val_pred, 'Confusion Matrix - Validation Set')
plot_confusion_matrix(y_test, y_test_pred, 'Confusion Matrix - Test Set')

In [ ]:
# Plot ROC curves
plot_roc_curve(y_train, y_train_prob, 'ROC Curve - Training Set')
plot_roc_curve(y_val, y_val_prob, 'ROC Curve - Validation Set')
plot_roc_curve(y_test, y_test_prob, 'ROC Curve - Test Set')

In [ ]:
# Plot precision-recall curves
plot_precision_recall_curve(y_train, y_train_prob, 'Precision-Recall Curve - Training Set')
plot_precision_recall_curve(y_val, y_val_prob, 'Precision-Recall Curve - Validation Set')
plot_precision_recall_curve(y_test, y_test_prob, 'Precision-Recall Curve - Test Set')

In [ ]:
# Plot feature importance
if hasattr(feature_engineer, 'feature_names'):
    feature_names = feature_engineer.feature_names
    
    # Get feature importance
    importance = best_xgb.feature_importances_
    
    # Create DataFrame for plotting
    feature_importance = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importance
    })
    
    # Sort by importance
    feature_importance = feature_importance.sort_values('Importance', ascending=False)
    
    # Plot top 20 features
    plt.figure(figsize=(12, 10))
    sns.barplot(x='Importance', y='Feature', data=feature_importance.head(20))
    plt.title('Top 20 Feature Importance')
    plt.tight_layout()
    plt.show()

## 7. Save the Model

In [ ]:
# Create models directory if it doesn't exist
os.makedirs(MODELS_DIR, exist_ok=True)

# Save the model
model_path = os.path.join(MODELS_DIR, 'churn_model.joblib')
joblib.dump(best_xgb, model_path)
print(f&quot;Model saved to {model_path}&quot;)

## 8. Conclusion and Next Steps

### 8.1 Summary of Model Development

In this notebook, we developed a machine learning model for predicting customer churn. The key steps included:

1. **Data Preparation**: We loaded and preprocessed the customer churn dataset.

2. **Feature Engineering**: We created additional features such as interaction features and service count features to improve model performance.

3. **Baseline Models**: We trained and evaluated several baseline models, including Logistic Regression, Random Forest, Gradient Boosting, XGBoost, LightGBM, and CatBoost.

4. **Handling Class Imbalance**: We applied SMOTE to address the class imbalance in the dataset, which improved the recall of our models.

5. **Hyperparameter Tuning**: We performed hyperparameter tuning on the XGBoost model using RandomizedSearchCV to find the optimal parameters.

6. **Model Evaluation**: We evaluated the final model using various metrics, including accuracy, precision, recall, F1 score, ROC AUC, and average precision.

7. **Visualization**: We visualized the model performance using confusion matrices, ROC curves, precision-recall curves, and feature importance plots.

### 8.2 Key Findings

1. **Model Performance**: The XGBoost model with optimized hyperparameters achieved good performance, with a ROC AUC score of around 0.85-0.87 on the test set.

2. **Important Features**: The most important features for predicting churn include contract type, tenure, monthly charges, and payment method.

3. **Class Imbalance**: SMOTE helped improve the recall of our models, which is important for identifying customers at risk of churning.

4. **Feature Engineering**: The additional features we created, such as service count and interaction features, contributed to the model's performance.

### 8.3 Next Steps

1. **Model Deployment**: Deploy the model as an API for real-time predictions.

2. **Model Monitoring**: Set up monitoring to track the model's performance over time and detect drift.

3. **Feature Engineering**: Explore additional feature engineering techniques to further improve model performance.

4. **Explainability**: Implement model explainability techniques such as SHAP values to better understand individual predictions.

5. **Business Integration**: Integrate the model with business processes to proactively identify and retain customers at risk of churning.